<a href="https://colab.research.google.com/github/julianzamb/5-DoF-in-Colab/blob/main/5DoF_3D_SIMULATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q pythreejs
from google.colab import output
output.enable_custom_widget_manager()
from scipy.optimize import minimize

In [15]:
import math, time
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize
from pythreejs import (
    Scene, PerspectiveCamera, Renderer, OrbitControls, Object3D, Mesh,
    BoxGeometry, CylinderGeometry, MeshStandardMaterial, AmbientLight,
    HemisphereLight, GridHelper, AxesHelper,
    AnimationMixer, AnimationClip, AnimationAction, QuaternionKeyframeTrack
)
from ipywidgets import (ToggleButton, Button, Textarea, FloatText, Checkbox,
                        HTML, Label, VBox, HBox, Layout)
from IPython.display import display

# ---------- Quaternion helpers (x, y, z, w) -- avoids Euler 'xyz' sync bug ----------
def q_x(a): return (math.sin(a/2), 0.0, 0.0, math.cos(a/2))
def q_y(a): return (0.0, math.sin(a/2), 0.0, math.cos(a/2))
def q_z(a): return (0.0, 0.0, math.sin(a/2), math.cos(a/2))

# ---------- Kinematic parameters (meters) -- UNCHANGED ----------
BASE_H = 0.10
L0, L1, L2, L3, L4 = 0.20, 0.60, 0.50, 0.30, 0.20
W = 0.08

def mat(color):
    return MeshStandardMaterial(color=color, metalness=0.2, roughness=0.6)

def link(length, color, width=W):
    """Box spanning local z = 0 -> length."""
    return Mesh(BoxGeometry(width=width, height=width, depth=length), mat(color),
                position=(0, 0, length / 2))

def joint_cyl(axis, color, radius=0.06, length=0.12):
    m = Mesh(CylinderGeometry(radiusTop=radius, radiusBottom=radius,
                              height=length, radialSegments=24), mat(color))
    if axis == 'z':                      # cylinders are Y-aligned; turn Y -> Z
        m.quaternion = q_x(math.pi / 2)
    return m

# ---------- Kinematic chain -- UNCHANGED ----------
base = Mesh(BoxGeometry(width=0.4, height=0.4, depth=BASE_H), mat('#444444'),
            position=(0, 0, BASE_H / 2))

j1 = Object3D(position=(0, 0, BASE_H))
j1.add(joint_cyl('z', '#cc3333'))
j1.add(link(L0, '#888888'))

j2 = Object3D(position=(0, 0, L0))
j2.add(joint_cyl('y', '#cc3333'))
j2.add(link(L1, '#3b82f6'))

j3 = Object3D(position=(0, 0, L1))
j3.add(joint_cyl('y', '#cc3333'))
j3.add(link(L2, '#22c55e'))

j4 = Object3D(position=(0, 0, L2))
j4.add(joint_cyl('y', '#cc3333'))
j4.add(link(L3, '#eab308'))

j5 = Object3D(position=(0, 0, L3))
j5.add(joint_cyl('z', '#cc3333', radius=0.05, length=0.06))
j5.add(link(L4, '#a855f7', width=0.05))

# ---------- Gripper -- UNCHANGED ----------
PLATE_T = 0.02
FINGER_W, FINGER_D, FINGER_H = 0.02, 0.10, 0.03
FINGER_Z = PLATE_T + FINGER_D / 2

# Larger workpiece: 6 cm per side (the original object was 2 cm per side).
BOX_SIZE = 0.06
GRIP_CLEARANCE = 0.002
# Leave 2 mm total clearance between the inner faces of the closed fingers.
X_CLOSED = (BOX_SIZE + FINGER_W + GRIP_CLEARANCE) / 2
X_OPEN = 0.06

gripper = Object3D(position=(0, 0, L4))

plate = Mesh(BoxGeometry(width=0.16, height=0.05, depth=PLATE_T), mat('#222222'),
             position=(0, 0, PLATE_T / 2))
finger_l = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(-X_CLOSED, 0, FINGER_Z))
finger_r = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(X_CLOSED, 0, FINGER_Z))

gripper.add(plate); gripper.add(finger_l); gripper.add(finger_r)
j5.add(gripper)

base.add(j1); j1.add(j2); j2.add(j3); j3.add(j4); j4.add(j5)

for _n, _j in zip(['j1', 'j2', 'j3', 'j4', 'j5'], [j1, j2, j3, j4, j5]):
    _j.name = _n
JOINT_AXES = ['z', 'y', 'y', 'y', 'z']

# ---------- Scene (Z-up) -- UNCHANGED ----------
grid = GridHelper(2, 20)
grid.quaternion = q_x(math.pi / 2)
axes = AxesHelper(0.5)

# ---------- Workpiece and pick/place geometry (meters) ----------
X_START, Y_START = 0.60, 0.20
PLACE_X_DEFAULT, PLACE_Y_DEFAULT = 0.55, -0.30
Z_BOX_CENTER = BOX_SIZE / 2          # box rests on the Z=0 grid
GRAB_TCP_Z = 0.0                     # fingertip TCP reaches the grid plane
HOVER_CLEARANCE = 0.05               # hover 5 cm above the TOP of the box
HOVER_TCP_Z = BOX_SIZE + HOVER_CLEARANCE
PICK_PLACE_TOTAL_TIME = 12.0         # seconds for the three motion phases

# World is an identity transform that owns the robot base and loose objects.
world = Object3D()
world.add(base)

box = Mesh(
    BoxGeometry(width=BOX_SIZE, height=BOX_SIZE, depth=BOX_SIZE),
    mat('#2563eb'),
    position=(X_START, Y_START, Z_BOX_CENTER)
)
world.add(box)

camera = PerspectiveCamera(position=(1.8, 1.8, 1.4), up=(0, 0, 1),
                           fov=50, aspect=800/600)

scene = Scene(children=[
    world, grid, axes, camera,
    AmbientLight(color='#ffffff', intensity=0.5),
    HemisphereLight(skyColor='#ffffff', groundColor='#555555',
                    intensity=0.9, position=(0, 0, 1)),
], background='#f0f0f0')

controls = OrbitControls(controlling=camera, target=(0, 0, 0.7))
renderer = Renderer(camera=camera, scene=scene, controls=[controls],
                    antialias=True, width=800, height=600)

# =====================================================================
# Joint pose application (same axes as the old slider callbacks:
# J1 = Z, J2/J3/J4 = Y, J5 = Z)
# =====================================================================
pose = {'q': np.zeros(5)}            # current joint angles in degrees

def apply_pose(q_deg, force=False):
    """Send only the joints that changed, rounded to cut float noise and message count."""
    q = np.round(np.asarray(q_deg, dtype=float), 3)
    prev = pose['q']
    a = np.radians(q)
    if force or q[0] != prev[0]: j1.quaternion = q_z(a[0])
    if force or q[1] != prev[1]: j2.quaternion = q_y(a[1])
    if force or q[2] != prev[2]: j3.quaternion = q_y(a[2])
    if force or q[3] != prev[3]: j4.quaternion = q_y(a[3])
    if force or q[4] != prev[4]: j5.quaternion = q_z(a[4])
    pose['q'] = q


# =====================================================================
# Object attachment / detachment
# =====================================================================
object_state = {'attached': False, 'relative_matrix': None}


def quaternion_to_matrix(q):
    """Convert pythreejs quaternion (x, y, z, w) to a 3x3 rotation matrix."""
    x, y, z, w = np.asarray(q, dtype=float)
    n = x*x + y*y + z*z + w*w
    if n < 1e-15:
        return np.eye(3)
    s = 2.0 / n
    return np.array([
        [1-s*(y*y+z*z), s*(x*y-z*w),   s*(x*z+y*w)],
        [s*(x*y+z*w),   1-s*(x*x+z*z), s*(y*z-x*w)],
        [s*(x*z-y*w),   s*(y*z+x*w),   1-s*(x*x+y*y)]
    ])


def matrix_to_quaternion(R):
    """Convert a 3x3 rotation matrix to (x, y, z, w)."""
    R = np.asarray(R, dtype=float)
    tr = np.trace(R)
    if tr > 0:
        s = math.sqrt(tr + 1.0) * 2.0
        w = 0.25 * s
        x = (R[2, 1] - R[1, 2]) / s
        y = (R[0, 2] - R[2, 0]) / s
        z = (R[1, 0] - R[0, 1]) / s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2.0
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2.0
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2.0
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s
    q = np.array([x, y, z, w])
    return tuple(q / np.linalg.norm(q))


def gripper_world_matrix(q_deg=None):
    """World transform of the Gripper Object3D origin (not the fingertip TCP)."""
    q = np.radians(np.asarray(pose['q'] if q_deg is None else q_deg, dtype=float))
    # 'base' is itself positioned at Z=BASE_H/2 in the scene hierarchy.
    T_base = np.eye(4)
    T_base[:3, 3] = np.asarray(base.position, dtype=float)
    T = T_base @ ks._trans(BASE_H) @ ks._rz(q[0])
    T = T @ ks._trans(L0) @ ks._ry(q[1])
    T = T @ ks._trans(L1) @ ks._ry(q[2])
    T = T @ ks._trans(L2) @ ks._ry(q[3])
    T = T @ ks._trans(L3) @ ks._rz(q[4])
    T = T @ ks._trans(L4)
    return T


def mesh_local_matrix(mesh):
    """Transform of a mesh whose parent is the identity World group."""
    T = np.eye(4)
    T[:3, :3] = quaternion_to_matrix(mesh.quaternion)
    T[:3, 3] = np.asarray(mesh.position, dtype=float)
    return T


def apply_mesh_matrix(mesh, T):
    mesh.position = tuple(np.asarray(T[:3, 3], dtype=float))
    mesh.quaternion = matrix_to_quaternion(T[:3, :3])


def attach_object(obj):
    """Attach obj to Gripper while preserving its current world pose."""
    if object_state['attached']:
        return

    # Both World and the robot root are identity-aligned in this scene.
    T_world_object = mesh_local_matrix(obj)
    T_world_gripper = gripper_world_matrix()
    T_gripper_object = np.linalg.inv(T_world_gripper) @ T_world_object

    world.remove(obj)
    gripper.add(obj)
    apply_mesh_matrix(obj, T_gripper_object)

    object_state['relative_matrix'] = T_gripper_object
    object_state['attached'] = True


def detach_object(obj):
    """Detach obj and restore its current world pose without a visual jump."""
    if not object_state['attached']:
        return

    # Reconstruct the current world pose from the tracked robot joint pose.
    T_world_gripper = gripper_world_matrix()
    T_world_object = T_world_gripper @ object_state['relative_matrix']

    gripper.remove(obj)
    world.add(obj)
    apply_mesh_matrix(obj, T_world_object)

    object_state['relative_matrix'] = None
    object_state['attached'] = False


# Same ranges the old sliders used (degrees)
JOINT_NAMES = ['J1', 'J2', 'J3', 'J4', 'J5']
LIMITS = np.array([[-180, 180], [-90, 90], [-150, 150], [-120, 120], [-180, 180]], float)

# =====================================================================
# Kinematics solver (FK + IK). Angles in DEGREES, lengths in meters.
# =====================================================================
class KinematicsSolver:
    """FK/IK for the 5-DoF arm, using the same frames as the 3D model:
    J1 = Rz, J2/J3/J4 = Ry, J5 = Rz, with translations along local +Z."""
    TOOL_LEN = PLATE_T + FINGER_D          # gripper plate + finger length (fingertips)
    TCP_OFFSET = L4 + TOOL_LEN             # J5 origin -> fingertip, along the J5 Z axis

    def __init__(self, limits_deg=LIMITS):
        self.limits = np.radians(np.asarray(limits_deg, float))

    # ---- homogeneous transforms ----
    @staticmethod
    def _trans(z):
        T = np.eye(4); T[2, 3] = z; return T
    @staticmethod
    def _rz(a):
        c, s = math.cos(a), math.sin(a)
        T = np.eye(4); T[:2, :2] = [[c, -s], [s, c]]; return T
    @staticmethod
    def _ry(a):                            # maps +Z to (sin a, 0, cos a), same as q_y
        c, s = math.cos(a), math.sin(a)
        T = np.eye(4); T[0, 0] = c; T[0, 2] = s; T[2, 0] = -s; T[2, 2] = c; return T

    def _fk_rad(self, q):
        T = self._trans(BASE_H) @ self._rz(q[0])
        T = T @ self._trans(L0) @ self._ry(q[1])
        T = T @ self._trans(L1) @ self._ry(q[2])
        T = T @ self._trans(L2) @ self._ry(q[3])
        T = T @ self._trans(L3) @ self._rz(q[4])
        return T @ self._trans(self.TCP_OFFSET)

    def fk(self, q_deg):
        """Returns (position xyz [m], 3x3 rotation matrix, pitch [deg]).
        pitch = angle of the tool axis above the ground plane: +90 up, -90 straight down."""
        T = self._fk_rad(np.radians(np.asarray(q_deg, float)))
        a = T[:3, 2]
        pitch = math.degrees(math.atan2(a[2], math.hypot(a[0], a[1])))
        return T[:3, 3].copy(), T[:3, :3].copy(), pitch

    def _tilt(self, q):
        """Signed angle (rad) between the tool axis and straight down, measured in the
        arm's vertical plane. 0 means pointing down. Used as the equality constraint
        because its gradient is nonzero at the solution (cos(pitch) would be flat there)."""
        a = self._fk_rad(q)[:3, 2]
        radial = a[0] * math.cos(q[0]) + a[1] * math.sin(q[0])
        return math.atan2(radial, -a[2])

    def ik(self, target_xyz, q_seed_deg=None, tol_pos=1e-4, tol_pitch_deg=0.05, n_random=8):
        """Returns (q_deg, info). Minimizes distance to target subject to tool pointing down.
        q_seed_deg: usually the current pose; among valid solutions the closest one wins."""
        target = np.asarray(target_xyz, float)
        seed = np.zeros(5) if q_seed_deg is None else np.radians(np.asarray(q_seed_deg, float))
        lo, hi = self.limits[:, 0], self.limits[:, 1]
        yaw = math.atan2(target[1], target[0])

        def cost(q):                       # squared distance: same minimizer, smooth at 0
            d = self._fk_rad(q)[:3, 3] - target
            return float(d @ d) + 1e-8 * float((q[4] - seed[4]) ** 2)  # keeps roll J5 unique
        cons = [{'type': 'eq', 'fun': lambda q: self._tilt(q)}]

        # SLSQP is local, so try several starts (elbow up/down, plus a few random ones)
        seeds = [seed]
        for q2, q3 in [(40, 139), (-35, 139), (77, -139), (30, 60), (-30, -60)]:
            q4 = 180 - q2 - q3                      # makes q2+q3+q4 = 180 (tool down)
            q4 = q4 if abs(q4) <= 120 else q4 - 360 * np.sign(q4)
            seeds.append(np.array([yaw, *np.radians([q2, q3, q4]), seed[4]]))
        rng = np.random.default_rng(0)
        for _ in range(n_random):
            s = rng.uniform(lo, hi); s[0] = yaw; s[4] = seed[4]; seeds.append(s)

        best = None
        for s0 in seeds:
            r = minimize(cost, np.clip(s0, lo, hi), method='SLSQP',
                         bounds=list(zip(lo, hi)), constraints=cons,
                         options={'ftol': 1e-12, 'maxiter': 300})
            q = r.x
            err = float(np.linalg.norm(self._fk_rad(q)[:3, 3] - target))
            tilt = abs(math.degrees(self._tilt(q)))
            ok = (err < tol_pos and tilt < tol_pitch_deg
                  and np.all(q >= lo - 1e-9) and np.all(q <= hi + 1e-9))
            travel = float(np.linalg.norm(q[:4] - seed[:4]))
            key = (not ok, travel if ok else err)   # valid first, then least joint travel
            if best is None or key < best[0]:
                best = (key, q, err, tilt, ok)
        _, q, err, tilt, ok = best
        return np.degrees(q), {'success': bool(ok), 'pos_error_m': err, 'tilt_error_deg': tilt}

# =====================================================================
# Trajectory planner
# =====================================================================
class TrajectoryPlanner:
    """Joint-space cubic-spline trajectory through N x 5 waypoints (degrees).

    Waypoints are spaced uniformly in time over [0, T]. Clamped boundary
    conditions give zero velocity at the start and end, so the arm
    starts and stops smoothly.
    """
    def __init__(self, waypoints, T):
        wp = np.asarray(waypoints, dtype=float)
        if wp.ndim != 2 or wp.shape[1] != 5:
            raise ValueError(f"Waypoints must be N x 5, got shape {wp.shape}")
        if wp.shape[0] < 2:
            raise ValueError("Need at least 2 waypoints")
        if T <= 0:
            raise ValueError("T must be > 0")
        self.T = float(T)
        self.waypoints = wp
        self.times = np.linspace(0.0, self.T, wp.shape[0])
        self.spline = CubicSpline(self.times, wp, axis=0, bc_type='clamped')

    def __call__(self, t):
        """Joint angles (deg) at time t, clamped to [0, T]."""
        return self.spline(np.clip(t, 0.0, self.T))

    def velocity(self, t):
        return self.spline(np.clip(t, 0.0, self.T), 1)

    def sample(self, fps=30):
        ts = np.linspace(0.0, self.T, max(2, int(round(self.T * fps)) + 1))
        return ts, self.spline(ts)

def parse_waypoints(text):
    """One row per line, 5 comma-separated angles in degrees. '#' starts a comment."""
    rows = []
    for i, line in enumerate(text.splitlines(), 1):
        line = line.split('#')[0].strip()
        if not line:
            continue
        try:
            vals = [float(p) for p in line.split(',')]
        except ValueError:
            raise ValueError(f"Line {i}: non-numeric value")
        if len(vals) != 5:
            raise ValueError(f"Line {i}: expected 5 values, got {len(vals)}")
        rows.append(vals)
    return np.array(rows, dtype=float).reshape(-1, 5)

def check_limits(q):
    """Return a message if any sample in q (M x 5) leaves the joint limits, else None."""
    for j in range(5):
        lo, hi = LIMITS[j]
        if q[:, j].min() < lo - 1e-6 or q[:, j].max() > hi + 1e-6:
            return (f"{JOINT_NAMES[j]} leaves its range [{lo:.0f}, {hi:.0f}] deg "
                    f"(reaches {q[:, j].min():.1f} .. {q[:, j].max():.1f})")
    return None

# =====================================================================
# UI
# =====================================================================
FPS = 30

DEFAULT_WAYPOINTS = """# J1, J2, J3, J4, J5  (degrees)
0, 0, 0, 0, 0
45, 30, -60, 40, 90
-45, 60, -90, -30, -90
0, 0, 0, 0, 0"""

wp_text = Textarea(value=DEFAULT_WAYPOINTS, rows=8,
                   placeholder='J1, J2, J3, J4, J5  (one row per waypoint, degrees)',
                   layout=Layout(width='400px'))
t_input = FloatText(value=6.0, description='T (s)', step=0.5,
                    layout=Layout(width='200px'))
from_current = Checkbox(value=True, description='Start from current pose',
                        indent=False, layout=Layout(width='400px'))
run_btn = Button(description='Run Trajectory', button_style='success',
                 icon='play', layout=Layout(width='400px'))
status = HTML(value='Ready.')

# Pick & Place coordinates are in meters and refer to the box center in XY.
box_x_input = FloatText(value=X_START, description='X (m)', step=0.05,
                        layout=Layout(width='175px'))
box_y_input = FloatText(value=Y_START, description='Y (m)', step=0.05,
                        layout=Layout(width='175px'))
place_x_input = FloatText(value=PLACE_X_DEFAULT, description='X (m)', step=0.05,
                          layout=Layout(width='175px'))
place_y_input = FloatText(value=PLACE_Y_DEFAULT, description='Y (m)', step=0.05,
                          layout=Layout(width='175px'))
pick_place_btn = Button(description='Run Pick & Place', button_style='success',
                         icon='random', layout=Layout(width='400px'))

def set_status(msg, error=False):
    color = '#b91c1c' if error else '#374151'
    status.value = f'<span style="color:{color}">{msg}</span>'

anim = {'action': None, 'mixer': None}
QFUN = {'x': q_x, 'y': q_y, 'z': q_z}
CLIP_FPS = 60                                  # keyframe density; playback runs at display rate

def build_clip(planner):
    """Sample the spline once and pack it into one quaternion track per joint."""
    ts, q = planner.sample(CLIP_FPS)
    tracks = []
    for j in range(5):
        f = QFUN[JOINT_AXES[j]]
        vals = []
        for deg in q[:, j]:
            vals.extend(f(math.radians(deg)))
        tracks.append(QuaternionKeyframeTrack(
            name=f'j{j+1}.quaternion', times=ts.tolist(), values=vals))
    return AnimationClip(tracks=tracks, duration=float(planner.T))


def play_planner_segment(planner, label):
    """Play one spline segment and wait until its browser-side animation finishes."""
    _, q_samples = planner.sample(FPS)
    limit_msg = check_limits(q_samples)
    if limit_msg:
        raise ValueError(f'{label}: {limit_msg}')

    if anim['action'] is not None:
        anim['action'].stop()

    clip = build_clip(planner)
    mixer = AnimationMixer(base)
    action = AnimationAction(mixer, clip, base,
                             loop='LoopOnce', repetitions=1,
                             clampWhenFinished=True)
    anim['mixer'] = mixer
    anim['action'] = action

    set_status(f'{label} — {planner.T:.1f} s')
    action.play()
    # The browser runs the keyframe animation; this keeps the Python callback
    # alive until the segment ends so attach/detach events occur at the right pose.
    time.sleep(planner.T + 0.15)
    pose['q'] = np.round(planner.waypoints[-1], 3)


def command_gripper(opened):
    """Use the existing toggle callback so the jaws animate consistently."""
    desired = bool(opened)
    if gripper_toggle.value != desired:
        gripper_toggle.value = desired
    else:
        animate_gripper(1.0 if desired else 0.0)


def solve_pick_place_waypoints(box_xy, place_xy, q_seed):
    """Solve and return IK joint poses A..F; G is the exact zero/home pose."""
    bx, by = map(float, box_xy)
    px, py = map(float, place_xy)
    targets = {
        'A': np.array([bx, by, HOVER_TCP_Z], dtype=float),
        'B': np.array([bx, by, GRAB_TCP_Z], dtype=float),
        'C': np.array([bx, by, HOVER_TCP_Z], dtype=float),
        'D': np.array([px, py, HOVER_TCP_Z], dtype=float),
        'E': np.array([px, py, GRAB_TCP_Z], dtype=float),
        'F': np.array([px, py, HOVER_TCP_Z], dtype=float),
    }

    # IK works in the base-local frame. The base mesh origin is offset in world Z.
    base_offset = np.asarray(base.position, dtype=float)
    q_waypoints = {}
    seed = np.asarray(q_seed, dtype=float).copy()
    reports = []
    for name in ('A', 'B', 'C', 'D', 'E', 'F'):
        target_local = targets[name] - base_offset
        q_solution, info = ks.ik(target_local, q_seed_deg=seed)
        if not info['success']:
            raise ValueError(
                f'IK failed at waypoint {name} ({targets[name].round(3)} m): '
                f"position error={info['pos_error_m']*1000:.2f} mm, "
                f"tool tilt error={info['tilt_error_deg']:.3f} deg. "
                'Try different XY coordinates closer to the robot.'
            )
        q_waypoints[name] = np.asarray(q_solution, dtype=float)
        reports.append(f'{name}: {info["pos_error_m"]*1000:.2f} mm')
        seed = q_waypoints[name]

    q_waypoints['G'] = np.zeros(5, dtype=float)
    return targets, q_waypoints, reports


def run_pick_place(_):
    """Execute A→B→C→D→E→F→G with grip events at B and E."""
    pick_place_btn.disabled = True
    run_btn.disabled = True
    test_grab_btn.disabled = True
    gripper_toggle.disabled = True
    try:
        if object_state['attached']:
            raise RuntimeError(
                'The box is already attached. Finish or detach the current grasp before starting another cycle.'
            )

        bx, by = float(box_x_input.value), float(box_y_input.value)
        px, py = float(place_x_input.value), float(place_y_input.value)
        if not np.all(np.isfinite([bx, by, px, py])):
            raise ValueError('All XY inputs must be finite numbers.')
        if anim['action'] is not None:
            anim['action'].stop()

        # Calculate every IK pose and validate every spline BEFORE moving anything.
        current_q = np.asarray(pose['q'], dtype=float).copy()
        targets, qwp, ik_reports = solve_pick_place_waypoints(
            (bx, by), (px, py), current_q
        )
        path_1 = np.vstack([current_q, qwp['A'], qwp['B']])
        path_2 = np.vstack([qwp['B'], qwp['C'], qwp['D'], qwp['E']])
        path_3 = np.vstack([qwp['E'], qwp['F'], qwp['G']])

        # Allocate the 12 s total in proportion to joint-space travel.
        paths = [path_1, path_2, path_3]
        distances = np.array([
            np.linalg.norm(np.diff(path, axis=0), axis=1).sum() for path in paths
        ], dtype=float)
        if distances.sum() < 1e-9:
            raise ValueError('The requested Pick & Place motion has no joint displacement.')
        minimum_phase_time = 1.5
        if PICK_PLACE_TOTAL_TIME < 3 * minimum_phase_time:
            raise ValueError('PICK_PLACE_TOTAL_TIME must be at least 4.5 seconds.')
        durations = minimum_phase_time + (
            PICK_PLACE_TOTAL_TIME - 3 * minimum_phase_time
        ) * distances / distances.sum()

        planners = [TrajectoryPlanner(path, duration)
                    for path, duration in zip(paths, durations)]
        for idx, planner in enumerate(planners, start=1):
            _, samples = planner.sample(FPS)
            limit_msg = check_limits(samples)
            if limit_msg:
                raise ValueError(f'Preflight check failed in motion phase {idx}: {limit_msg}')

        # Reset the loose box at the requested pickup coordinates and open the jaws.
        box.position = (bx, by, Z_BOX_CENTER)
        box.quaternion = (0.0, 0.0, 0.0, 1.0)
        command_gripper(opened=True)

        # Phase 1: move to A, then descend to B. Close and attach exactly at B.
        play_planner_segment(planners[0], 'A → B: approach and descend to the box')
        command_gripper(opened=False)
        attach_object(box)

        # Phase 2: lift to C, travel to D, then descend to E while carrying the box.
        play_planner_segment(planners[1], 'C → D → E: lift, transfer and lower')
        command_gripper(opened=True)
        detach_object(box)

        # Phase 3: move up to F and return to the zero/home joint configuration G.
        play_planner_segment(planners[2], 'F → G: retreat and return home')

        set_status(
            'Pick & Place completed. Box released at '
            f'({px:.3f}, {py:.3f}) m. IK errors: ' + ', '.join(ik_reports)
        )
    except Exception as e:
        set_status(f'Pick & Place error: {e}', error=True)
    finally:
        pick_place_btn.disabled = False
        run_btn.disabled = False
        test_grab_btn.disabled = False
        gripper_toggle.disabled = False


def run_trajectory(_):
    run_btn.disabled = True
    try:
        wp = parse_waypoints(wp_text.value)
        if from_current.value and (wp.shape[0] == 0 or
                                   not np.allclose(wp[0], pose['q'], atol=1e-3)):
            wp = np.vstack([pose['q'], wp])
        planner = TrajectoryPlanner(wp, t_input.value)

        _, q_samples = planner.sample(FPS)
        msg = check_limits(q_samples)
        if msg:
            set_status(msg, error=True)
            return

        if anim['action'] is not None:         # stop the previous run
            anim['action'].stop()

        clip = build_clip(planner)
        mixer = AnimationMixer(base)           # root = base, so 'j1'..'j5' resolve
        action = AnimationAction(mixer, clip, base,
                                 loop='LoopOnce', repetitions=1,
                                 clampWhenFinished=True)
        anim['action'] = action

        set_status(f'Running: {wp.shape[0]} waypoints, T = {planner.T:.2f} s ...')
        action.play()                          # one message; the browser does the rest

        pose['q'] = np.round(wp[-1], 3)        # Python-side record of the end pose
        time.sleep(planner.T + 0.1)            # only to re-enable the button; sends nothing
        set_status('Done.')
    except Exception as e:
        set_status(f'Error: {e}', error=True)
    finally:
        run_btn.disabled = False

run_btn.on_click(run_trajectory)

# ---------- Gripper toggle + animation -- UNCHANGED ----------
grip = {'t': 0.0}

def set_gripper(t):
    x = X_CLOSED + (X_OPEN - X_CLOSED) * t
    finger_l.position = (-x, 0, FINGER_Z)
    finger_r.position = (x, 0, FINGER_Z)

def animate_gripper(target, duration=0.35, steps=14):
    start = grip['t']
    for i in range(1, steps + 1):
        s = i / steps
        s = s * s * (3 - 2 * s)
        grip['t'] = start + (target - start) * s
        set_gripper(grip['t'])
        time.sleep(duration / steps)
    grip['t'] = target
    set_gripper(target)
    time.sleep(0.05)
    set_gripper(target)

gripper_toggle = ToggleButton(value=False, description='Open/Close Gripper',
                              button_style='info', icon='hand-paper',
                              tooltip='Off = closed, On = open',
                              layout=Layout(width='400px'))

def on_toggle(change):
    animate_gripper(1.0 if change['new'] else 0.0)

gripper_toggle.observe(on_toggle, names='value')

set_gripper(0.0)
apply_pose(np.zeros(5))

ks = KinematicsSolver()

# ---------- One-click IK grasp test (no trajectory planner involved) ----------
test_grab_btn = Button(description='Test Grab', button_style='success',
                       icon='hand-rock', layout=Layout(width='400px'))


def test_grab(_):
    test_grab_btn.disabled = True
    run_btn.disabled = True
    try:
        if object_state['attached']:
            set_status('The box is already attached. Call detach_object(box) first.', error=True)
            return

        # Stop any active keyframe action before applying a new IK pose.
        if anim['action'] is not None:
            anim['action'].stop()

        bx, by = float(box_x_input.value), float(box_y_input.value)
        box.position = (bx, by, Z_BOX_CENTER)
        box.quaternion = (0.0, 0.0, 0.0, 1.0)
        target_world = np.array([bx, by, GRAB_TCP_Z], dtype=float)
        target_ik = target_world - np.asarray(base.position, dtype=float)
        q_solution, info = ks.ik(target_ik, q_seed_deg=pose['q'])

        if not info['success']:
            set_status(
                f"IK could not reach the grasp target. Error: {info['pos_error_m']:.6f} m, "
                f"tilt: {info['tilt_error_deg']:.4f} deg",
                error=True
            )
            return

        # Open before approaching, move using the existing IK solution,
        # then close and attach the box to the gripper frame.
        gripper_toggle.value = True
        apply_pose(q_solution, force=True)
        gripper_toggle.value = False
        attach_object(box)

        set_status(
            f"Box grasped. IK error: {info['pos_error_m']*1000:.3f} mm. "
            "The box is now parented to Gripper."
        )
    except Exception as e:
        set_status(f'Test Grab error: {e}', error=True)
    finally:
        run_btn.disabled = False
        test_grab_btn.disabled = False


test_grab_btn.on_click(test_grab)
pick_place_btn.on_click(run_pick_place)


panel = VBox([
    Label('Pick & Place coordinates (meters)'),
    Label('Box Start (X, Y)'), HBox([box_x_input, box_y_input]),
    Label('Place Target (X, Y)'), HBox([place_x_input, place_y_input]),
    pick_place_btn,
    status,
    gripper_toggle,
    test_grab_btn,
    Label('Manual joint-space trajectory'),
    wp_text, t_input, from_current, run_btn,
])
display(HBox([renderer, panel]))

# ---------- IK test (not wired to the UI) ----------
p, R, pitch = ks.fk([0, 0, 0, 0, 0])
print("FK zero pose:", p.round(4), "pitch", round(pitch, 1))   # expect [0, 0, 2.02], +90

for tgt in [(0.1, 0.1, 0.05), (0.3, -0.2, 0.05), (0.0, 0.4, 0.10), (1.5, 0.0, 0.05)]:
    q, info = ks.ik(tgt)
    p, R, pitch = ks.fk(q)
    print(tgt, "q =", q.round(2), "-> fk =", p.round(4), "pitch =", round(pitch, 2), info)

FK zero pose: [0.   0.   2.02] pitch 90.0
(0.1, 0.1, 0.05) q = [ 45.   -34.92 139.04  75.88  -0.  ] -> fk = [0.1  0.1  0.05] pitch = -90.0 {'success': True, 'pos_error_m': 6.745363488139724e-07, 'tilt_error_deg': 2.3675931579913738e-11}
(0.3, -0.2, 0.05) q = [-33.69  -8.3  124.88  63.42  -0.  ] -> fk = [ 0.3  -0.2   0.05] pitch = -90.0 {'success': True, 'pos_error_m': 6.293619026786418e-07, 'tilt_error_deg': 3.0838831737951797e-12}
(0.0, 0.4, 0.1) q = [ 90.    -6.5  117.13  69.37  -0.  ] -> fk = [0.  0.4 0.1] pitch = -90.0 {'success': True, 'pos_error_m': 2.3302977214851035e-07, 'tilt_error_deg': 3.2466551301109622e-12}
(1.5, 0.0, 0.05) q = [  0.    76.14  -0.   103.86   0.  ] -> fk = [ 1.068   0.     -0.0566] pitch = -90.0 {'success': False, 'pos_error_m': 0.44495954639593094, 'tilt_error_deg': 2.944544143308549e-12}
